# ResNet-18 Transfer Learning: CIFAR-10 Embeddings, PCA, and t-SNE

This notebook completes the representation/transfer-learning lab.

The lab asks for:
1. Loading a pretrained ResNet-18 model.
2. Loading CIFAR-10 test images using ResNet/ImageNet preprocessing.
3. Extracting 512-dimensional penultimate-layer embeddings using a forward hook on `avgpool`.
4. Reducing embeddings to 2D using PCA and t-SNE.
5. Reducing embeddings to 3D using t-SNE and visualizing interactively.
6. Interpreting class separability and transfer learning quality.

Run all cells top to bottom and save the notebook with outputs before submitting.


## Runtime note

The assignment asks for the entire CIFAR-10 test set. This notebook uses all 10,000 test images by default.

t-SNE can be slow on CPU. If your machine is too slow, set `TSNE_N = 3000` for testing, then switch back to `10000` for final execution if possible.


In [ ]:
# Optional dependency installer
import sys
import subprocess
import importlib.util

packages = {
    "torch": "torch torchvision",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "plotly": "plotly"
}

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pip_name.split()])

print("Dependency check complete.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import plotly.express as px

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
BATCH_SIZE = 256
TSNE_N = 10000
RANDOM_STATE = 42

cifar10_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]


## Task 0: Load a pretrained ResNet-18 model

We load ResNet-18 pretrained on ImageNet and set it to evaluation mode. The model is used as a fixed feature extractor, not trained on CIFAR-10.


In [ ]:
try:
    weights = models.ResNet18_Weights.DEFAULT
    resnet18 = models.resnet18(weights=weights)
except Exception:
    print("Using older torchvision fallback: pretrained=True")
    resnet18 = models.resnet18(pretrained=True)

resnet18 = resnet18.to(device)
resnet18.eval()

print(resnet18)


### Interpretation

The pretrained ResNet-18 model is in evaluation mode, which fixes batch normalization behavior and disables training-specific updates. Because the model was trained on ImageNet, it contains general visual features that may transfer to CIFAR-10 without task-specific training.


## Task 1: Load CIFAR-10 test data with ImageNet preprocessing

ResNet-18 expects 224 x 224 RGB images normalized with ImageNet statistics:
- mean = `[0.485, 0.456, 0.406]`
- std = `[0.229, 0.224, 0.225]`


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

cifar10_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

test_loader = DataLoader(
    cifar10_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

x0, y0 = cifar10_test[0]

print("Number of test samples:", len(cifar10_test))
print("First input dtype:", x0.dtype)
print("First label dtype:", type(y0).__name__)
print("First input shape:", tuple(x0.shape))
print("First label:", y0, cifar10_classes[y0])
print("Min pixel after normalization:", float(x0.min()))
print("Max pixel after normalization:", float(x0.max()))
print("Number of channels:", x0.shape[0])


### Interpretation

The CIFAR-10 test set contains 10,000 images. Each image is resized to `(3, 224, 224)`, which matches the RGB input format expected by ResNet-18. Normalizing with ImageNet statistics is necessary because the pretrained model learned under that input distribution.


## Task 2: Extract 512-dimensional penultimate-layer embeddings

We register a forward hook on `resnet18.avgpool`. This layer produces the representation immediately before the final classifier. After flattening, each image has a 512-dimensional embedding.


In [ ]:
embedding_batches = []
label_batches = []

def hook_fn(module, input, output):
    embedding_batches.append(output.detach().cpu().flatten(start_dim=1))

hook_handle = resnet18.avgpool.register_forward_hook(hook_fn)

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        _ = resnet18(xb)
        label_batches.append(yb.cpu())

hook_handle.remove()

embeddings = torch.cat(embedding_batches, dim=0).numpy()
labels = torch.cat(label_batches, dim=0).numpy()
label_names = np.array([cifar10_classes[i] for i in labels])

print("Embeddings shape:", embeddings.shape)
print("Labels shape:", labels.shape)
print("Embedding dimension:", embeddings.shape[1])


### Interpretation

The embedding matrix should have shape `(10000, 512)`. This means every CIFAR-10 test image is represented by a 512-dimensional visual feature vector. These features are not CIFAR-10-specific; they come from the ImageNet-pretrained ResNet-18 representation space.


## Task 3: PCA and t-SNE in 2D

We reduce the 512-dimensional embeddings to 2D using PCA and t-SNE, then color points by class label.


In [ ]:
n_points = min(TSNE_N, embeddings.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
subset_idx = rng.choice(embeddings.shape[0], size=n_points, replace=False)

embeddings_subset = embeddings[subset_idx]
labels_subset = labels[subset_idx]
label_names_subset = label_names[subset_idx]

print("Visualization points:", n_points)


In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
emb_pca_2d = pca_2d.fit_transform(embeddings_subset)

print("PCA explained variance ratio:", pca_2d.explained_variance_ratio_)
print("Total explained variance:", pca_2d.explained_variance_ratio_.sum())


In [ ]:
tsne_2d = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE,
    max_iter=1000
)

emb_tsne_2d = tsne_2d.fit_transform(embeddings_subset)
print("2D t-SNE shape:", emb_tsne_2d.shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for class_id, class_name in enumerate(cifar10_classes):
    mask = labels_subset == class_id
    axes[0].scatter(emb_pca_2d[mask, 0], emb_pca_2d[mask, 1], s=8, alpha=0.65, label=class_name)

axes[0].set_title("PCA projection of ResNet-18 embeddings")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

for class_id, class_name in enumerate(cifar10_classes):
    mask = labels_subset == class_id
    axes[1].scatter(emb_tsne_2d[mask, 0], emb_tsne_2d[mask, 1], s=8, alpha=0.65, label=class_name)

axes[1].set_title("t-SNE projection of ResNet-18 embeddings")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

handles, legend_labels = axes[1].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="center right", bbox_to_anchor=(1.08, 0.5))
plt.tight_layout()
plt.show()


### Interpretation

PCA gives a linear projection of the 512-dimensional embeddings. It may show broad structure, but substantial overlap is expected because only two linear components cannot preserve all class-related information.

t-SNE usually reveals more local cluster structure. Classes with similar visual content may appear near each other, such as automobile and truck, or cat and dog. More visually distinctive categories may separate more clearly. This suggests that ImageNet-pretrained ResNet-18 learns transferable visual features that contain meaningful CIFAR-10 class information, even without training on CIFAR-10.


## Task 4: 3D t-SNE interactive visualization

Now we project the embeddings to three dimensions with t-SNE and plot them interactively.


In [ ]:
tsne_3d = TSNE(
    n_components=3,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE,
    max_iter=1000
)

emb_tsne_3d = tsne_3d.fit_transform(embeddings_subset)
print("3D t-SNE shape:", emb_tsne_3d.shape)


In [ ]:
tsne3d_df = pd.DataFrame({
    "tsne_1": emb_tsne_3d[:, 0],
    "tsne_2": emb_tsne_3d[:, 1],
    "tsne_3": emb_tsne_3d[:, 2],
    "label": labels_subset,
    "class_name": label_names_subset
})

fig = px.scatter_3d(
    tsne3d_df,
    x="tsne_1",
    y="tsne_2",
    z="tsne_3",
    color="class_name",
    opacity=0.75,
    title="3D t-SNE of ResNet-18 CIFAR-10 embeddings",
    width=900,
    height=700
)

fig.update_traces(marker=dict(size=3))
fig.show()


### Interpretation

The 3D t-SNE plot provides an additional dimension for inspecting cluster structure. When rotating the plot, class groups may become easier to separate than in 2D. The spatial arrangement shows that ResNet-18 captures general visual similarities such as object shape, texture, and semantic category.

The clusters are not perfectly separated because the model was pretrained on ImageNet rather than CIFAR-10, and CIFAR-10 images are smaller and less detailed. Still, visible grouping by class indicates that the pretrained embeddings transfer well to a new natural-image dataset.


## Final conclusion

This lab demonstrates transfer learning through representation extraction. A ResNet-18 model pretrained on ImageNet can produce meaningful 512-dimensional embeddings for CIFAR-10 test images without task-specific training. PCA and t-SNE show that these embeddings contain class-related structure. The results support the idea that ImageNet pretraining learns general visual representations that can transfer to other image datasets.


## Submission checklist

Before submitting:
- Run all cells.
- Confirm the embedding shape is `(10000, 512)` if using the full test set.
- Confirm the 2D PCA/t-SNE plots display.
- Confirm the interactive 3D t-SNE plot displays.
- Save the notebook with outputs visible.
